# Classification Metrics · Should this email go to spam?

An email arrives. Sending spam to the inbox can be distracting. Moving legitimate mail into the spam folder can hide something important.

| The email arrives | The filter chooses a folder | We check the decision |
| --- | --- | --- |
| The model can read numeric features such as the frequency of “free” and the longest capital-letter run. | At or above the cutoff → predict **Spam**, put it in the spam folder. Below the cutoff → predict **Legitimate**, leave it in the inbox. | Open the recorded label and add one count to the confusion matrix. |

**Positive (1)** means Spam. **Negative (0)** means Legitimate. A logistic regression model produces a Spam probability estimate from 0 to 1. Higher scores rank a message as more likely to be Spam. Treat the values as scores when choosing a cutoff. Check calibration on representative mail before interpreting a score of 0.5177 as a real-world 51.77% chance.

Run this notebook from top to bottom. First follow two emails, then use all 20, compare cutoffs and inspect the ROC curve. All cases come from the same fitted model and held-out set.

<details><summary>Where the emails, labels, and scores come from</summary>

UCI’s Spambase data contains numeric measurements and a recorded Spam label for each email. Of the 4,601 records, we use 3,433 to fit a standard scaler and logistic regression and reserve a 1,168-email fold for evaluation. Identical 57-feature vectors stay in the same fold. We use the same split as the full Bagging notebook. The scores below are the resulting Spam predictions rounded to four decimals, and the comparisons use these rounded scores. We select 10 held-out emails from each label, spread over the score ranks, so the matrix and AUC are small enough to count in class. **E4130** denotes source row 4,130. The last section reports the complete fold and shows how to reproduce the model.

[Spambase, Hopkins et al. (1999), UCI Machine Learning Repository](https://doi.org/10.24432/C53G6X) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

</details>

In [ ]:
# @title Load scores and recorded Spambase labels { display-mode: "form" }
# Each row contains (UCI source row ID, model Spam score, recorded label).
emails = [
    ("E1772", 1.0000, 1),
    ("E4320", 1.0000, 0),
    ("E613", 0.9998, 1),
    ("E1014", 0.9973, 1),
    ("E1729", 0.9925, 1),
    ("E1095", 0.9766, 1),
    ("E823", 0.9394, 1),
    ("E1365", 0.8585, 1),
    ("E994", 0.7561, 1),
    ("E1809", 0.5359, 1),
    ("E4130", 0.5177, 0),
    ("E4447", 0.2220, 0),
    ("E4491", 0.1523, 0),
    ("E2003", 0.0476, 0),
    ("E2187", 0.0155, 0),
    ("E4415", 0.0023, 0),
    ("E35", 0.0009, 1),
    ("E4335", 0.0001, 0),
    ("E1839", 0.0000, 0),
    ("E3058", 0.0000, 0),
]
baseline_cutoff = 0.50

## 1. Run the decision rule

At **0.50**, what happens to **E4130 (0.5177)** and **E35 (0.0009)**? Predict the folders before running the cell. Then set `cutoff` to **0.5177** and **0.5178**, and watch E4130 switch. Every comparison includes equality. The recorded label is read only when evaluating predictions.

In [ ]:
cutoff = 0.50
scores = [score for _, score, _ in emails]
actual = [label for _, _, label in emails]
predicted = [int(score >= cutoff) for score in scores]
correct = sum(p == y for p, y in zip(predicted, actual))

for (email, score, _), p in zip(emails, predicted):
    if email in {"E4130", "E35"}:
        print(f"{email}: {score:.4f} → {'Spam folder' if p else 'Inbox'}")
print(f"{correct} of {len(actual)} correct ({correct / len(actual):.0%})")

## 2. Open the actual label and name the cell

Read one email’s score and the folder decision. Open its recorded label to see where it lands. The rows represent **actual** labels and the columns represent **predicted** labels. Each email adds one count.

In [ ]:
# @title Reveal the decision, label and matrix cell { display-mode: "form" }
from html import escape
from IPython.display import HTML, display

style = '''
<style>
.spam-teaching { padding: 16px; border: 1px solid #cadcd5; border-radius: 12px;
 color: #173430; background: #f1f7f4; font: 16px/1.55 system-ui, sans-serif; }
.spam-teaching h3, .spam-teaching p { margin: 0 0 10px; }
.spam-teaching details { padding: 10px 12px; margin: 7px 0; border: 1px solid #cadcd5;
 border-radius: 8px; background: #fff; }
.spam-teaching summary { cursor: pointer; font-weight: 700; }
.spam-teaching details > div { margin-top: 10px; }
.spam-teaching table { width: 100%; border-collapse: collapse; margin: 8px 0; text-align: left; }
.spam-teaching :is(th, td) { padding: 7px; border-bottom: 1px solid #cadcd5; }
.spam-teaching .hit { background: #d5eee5; color: #134e41; font-weight: bold; }
.spam-teaching .figure { overflow-x: auto; }
.spam-teaching svg { display: block; width: 100%; max-width: 560px; min-width: 300px; }
</style>
'''

def panel(title, content):
    display(HTML(style + f'<section class="spam-teaching"><h3>{escape(title)}</h3>{content}</section>'))

def disclosure(title, content, group=None, opened=False):
    name = '' if group is None else f' name="{escape(group)}"'
    state = ' open' if opened else ''
    return f'<details{name}{state}><summary>{escape(title)}</summary><div>{content}</div></details>'

def class_label(actual):
    return 'Spam' if actual else 'Legitimate'

def folder(predicted):
    return 'Spam folder' if predicted else 'Inbox'

def outcome(y, p):
    if p:
        return 'TP' if y else 'FP'
    return 'FN' if y else 'TN'

def matrix(counts=None, highlighted=()):
    cells = {}
    for kind in ('TP', 'FN', 'FP', 'TN'):
        value = kind if counts is None else f'{counts[kind]} {kind}'
        cell_class = ' class="hit"' if kind in highlighted else ''
        cells[kind] = f'<td{cell_class}>{value}</td>'
    return (
        '<table><thead><tr><th scope="col">Recorded label</th><th scope="col">Predicted Spam (P)</th>'
        '<th scope="col">Predicted Legitimate (N)</th></tr></thead>'
        f'<tbody><tr><th scope="row">Spam (P)</th>{cells["TP"]}{cells["FN"]}</tr>'
        f'<tr><th scope="row">Legitimate (N)</th>{cells["FP"]}{cells["TN"]}</tr></tbody></table>'
    )

def trace_email(email, opened=False):
    identifier, score, y = email
    p = int(score >= cutoff)
    decision = f'<p>{score:.4f} {"≥" if p else "&lt;"} {cutoff:.4f} → <strong>{folder(p)}</strong>.</p>'
    observed = f'<p>The recorded label is <strong>{class_label(y)}</strong>. This adds one {outcome(y, p)}.</p>'
    label = disclosure('Show recorded label', observed + matrix(highlighted=(outcome(y, p),)))
    return disclosure(f'{identifier} · Spam score {score:.4f}', decision + label, 'spam-trace', opened)

trace_ids = ('E4130', 'E35')
trace_rows = [next(row for row in emails if row[0] == identifier) for identifier in trace_ids]
panel(
    f'Follow one email at cutoff {cutoff:.4f}',
    '<p>Read the score and prediction first. Open the label when you are ready to count it.</p>'
    + ''.join(trace_email(row, opened=i == 0) for i, row in enumerate(trace_rows)),
)

## 3. Count all 20 emails

For this example, reset to **0.50**. Point to the green cells in turn.

- **Precision** starts in the predicted Spam column. Of all 11 flagged emails, 9 really are Spam. Precision = 9/11 ≈ **81.82%**.
- **Recall** starts in the actual Spam row. Of all 10 Spam emails, 9 are caught. Recall = 9/10 = **90%**.
- **Accuracy** starts with both correct cells. (9 + 8)/20 = **85%**. A rule that predicts one label for all 20 gets 10/20 on this balanced selection.
- **F1** uses the harmonic mean of precision and recall. From the counts, 2TP/(2TP + FP + FN) = 18/(18 + 2 + 1) ≈ **85.71%**. The harmonic mean weights weak performance more strongly than a simple average. If precision is 100% and recall is 25%, the simple average is 62.5% and F1 is 40%.

**Reading the letters** · True and false say whether the prediction was correct. Positive and negative say which class the filter predicted.

In [ ]:
def measure(records, cutoff):
    counts = dict.fromkeys(('TP', 'FN', 'FP', 'TN'), 0)
    by_email = {}
    for email, score, y in records:
        cell = outcome(y, int(score >= cutoff))
        counts[cell] += 1
        by_email[email] = cell

    tp, fn, fp, tn = (counts[key] for key in ('TP', 'FN', 'FP', 'TN'))
    rates = {
        'Precision': tp / (tp + fp) if tp + fp else None,
        'Recall': tp / (tp + fn),
        'Accuracy': (tp + tn) / len(records),
        'F1': 2 * tp / (2 * tp + fp + fn),
        'FPR': fp / (fp + tn),
    }
    return counts, rates, by_email

def rate_text(value):
    return 'Undefined' if value is None else f'{value:.2%}'

base_counts, base_rates, base_cells = measure(emails, baseline_cutoff)
formulas = {
    'Precision': (('TP', 'FP'), 'Read the predicted Spam column.'),
    'Recall': (('TP', 'FN'), 'Read the row with the recorded Spam label.'),
    'Accuracy': (('TP', 'TN'), 'Read the two correct cells.'),
    'F1': (('TP', 'FP', 'FN'), 'Count TP twice, plus FP and FN, in the denominator.'),
}
panel(
    'Pick a metric and highlight its cells',
    ''.join(
        disclosure(
            f'{metric} · {rate_text(base_rates[metric])}',
            f'<p>{sentence}</p>' + matrix(base_counts, cells),
            'spam-metric',
            opened=metric == 'Precision',
        )
        for metric, (cells, sentence) in formulas.items()
    ),
)

## 4. Compare two cutoffs on the same emails

At **0.75**, two emails move to the inbox. **E4130 (Legitimate, 0.5177)** becomes a true negative. **E1809 (Spam, 0.5359)** becomes a false negative. Accuracy stays 85% while precision rises and recall falls. At **0.20**, **E4447 (Legitimate, 0.2220)** moves from true negative to false positive instead.

If sending legitimate mail to spam costs 10 penalty points and missing spam costs 1, the teaching cost is `10 × FP + FN`. The 0.50 cutoff costs 21 and the 0.75 cutoff costs 12. Pick model or cutoff settings on validation data using the error costs or constraints for your application. Then measure on an independent test set.

In [ ]:
# @title Open different cutoff policies { display-mode: "form" }
def comparison_card(other):
    counts, rates, cells = measure(emails, other)
    names = ('TP', 'FN', 'FP', 'TN', 'Precision', 'Recall', 'Accuracy', 'F1')
    table_rows = ''.join(
        f'<tr><th scope="row">{name}</th>'
        f'<td>{base_counts[name] if name in base_counts else rate_text(base_rates[name])}</td>'
        f'<td>{counts[name] if name in counts else rate_text(rates[name])}</td></tr>'
        for name in names
    )
    changes = [
        f'<li><strong>{email}</strong> · score {score:.4f} · actual {class_label(y)} · '
        f'{base_cells[email]} → {cells[email]}</li>'
        for email, score, y in emails if base_cells[email] != cells[email]
    ]
    change_text = f'<p>{len(changes)} emails change folders. Their recorded labels and scores stay fixed.</p>'
    change_text += '<ul>' + ''.join(changes) + '</ul>'
    table = (
        f'<table><thead><tr><th>Measure</th><th>Cutoff {baseline_cutoff:.2f}</th>'
        f'<th>Cutoff {other:.2f}</th></tr></thead><tbody>{table_rows}</tbody></table>'
    )
    if rates['Precision'] is None:
        change_text += '<p>Precision is undefined. Every email goes to the inbox, so the predicted Spam column is empty.</p>'
    return disclosure(f'Compare with cutoff {other:.2f}', table + change_text, 'spam-cutoff', other == 0.75)

panel('Same scores, different folders', ''.join(comparison_card(t) for t in (0.75, 0.20, 0.00, 1.01)))

## 5. Turn one matrix into one ROC point

The **receiver operating characteristic (ROC)** plots recall against false-positive rate. For the 0.50 cutoff, read the matrix:

- Horizontal · FPR = FP/(FP + TN) = 2/10 = **0.20**. How many legitimate emails were flagged?
- Vertical · TPR = TP/(TP + FN) = 9/10 = **0.90**. How much Spam did we catch? TPR is recall.

Start above the highest score. The filter flags no emails, so the point is (0, 0). Lower the cutoff to 1.0000. E1772 (Spam) moves it up by 1/10. E4320 (Legitimate) moves it right by 1/10. Those tied emails enter together. Then at 0.9998, E613 (Spam) adds another 1/10 vertically. Continue down the sorted list of distinct scores until everyone is flagged at (1, 1).

In [ ]:
def operating_point(cutoff):
    _, rates, _ = measure(emails, cutoff)
    return rates['FPR'], rates['Recall']

thresholds = [float('inf')] + sorted({score for _, score, _ in emails}, reverse=True)
points = [operating_point(t) for t in thresholds]
roc_area = sum(
    (right - left) * (bottom + top) / 2
    for (left, bottom), (right, top) in zip(points, points[1:])
)
print(f'ROC AUC from trapezoids: {roc_area:.3f}')

In [ ]:
# @title Move the cutoff marker along the curve { display-mode: "form" }
def roc_picture(cutoff):
    fpr, tpr = operating_point(cutoff)

    def x(rate):
        return 70 + 395 * rate

    def y(rate):
        return 250 - 215 * rate

    curve = ' '.join(f'{x(a):.1f},{y(b):.1f}' for a, b in points)
    ticks = ''.join(
        f'<text x="{x(v)}" y="270" text-anchor="middle">{v:g}</text>'
        f'<text x="60" y="{y(v) + 5}" text-anchor="end">{v:g}</text>'
        for v in (0, 0.25, 0.5, 0.75, 1)
    )
    return f'''
<div class="figure"><svg viewBox="0 0 500 300" role="img"
  aria-label="ROC at cutoff {cutoff:.2f}. False-positive rate {fpr:.0%}, recall {tpr:.0%}."
  style="font: 13px system-ui; fill: #173430">
  <path d="M70 35V250H465" stroke="#6c8d81" fill="none"/>
  <path d="M70 250L465 35" stroke="#6c8d81" stroke-dasharray="5 4" fill="none"/>
  <polyline points="{curve}" stroke="#087669" stroke-width="3" fill="none"/>
  <circle cx="{x(fpr)}" cy="{y(tpr)}" r="6" fill="#bd604c" stroke="#fff" stroke-width="2"/>
  {ticks}<text x="267" y="296" text-anchor="middle">False-positive rate</text>
  <text transform="translate(19 145) rotate(-90)" text-anchor="middle">Recall (TPR)</text>
</svg></div>
<p>Cutoff <strong>{cutoff:.2f}</strong> gives FPR <strong>{fpr:.0%}</strong> and recall <strong>{tpr:.0%}</strong>.</p>
'''

panel(
    'Inspect a cutoff on the full ROC curve',
    '<p>The orange point moves with the cutoff. The green curve and its area stay fixed.</p>'
    + ''.join(disclosure(f'Cutoff {t:.2f}', roc_picture(t), 'spam-roc', t == 0.50)
              for t in (1.01, 0.50, 0.75, 0.20, 0.00)),
)

## 6. Rebuild AUC by comparing pairs

AUC is the **area under the ROC curve**. The same number also measures how well Spam ranks above Legitimate mail overall. A Spam email earns **1** when its score is higher than a legitimate one, **0.5** for a tie, and **0** for a lower score.

| Actual Spam | Actual Legitimate | Credit |
| --- | --- | --- |
| E1809 at 0.5359 | E4130 at 0.5177 | 1 |
| E35 at 0.0009 | E4130 at 0.5177 | 0 |
| E1772 at 1.0000 | E4320 at 1.0000 | 0.5 |

A cutoff changes which emails you flag. The ranking comparisons use the original scores, so the AUC is the same at each cutoff.

In [ ]:
# @title Show pair count contributions by email { display-mode: "form" }
positives = [(email, score) for email, score, label in emails if label]
negatives = [(email, score) for email, score, label in emails if not label]
comparisons = []
for email, negative_score in negatives:
    wins = sum(score > negative_score for _, score in positives)
    ties = sum(score == negative_score for _, score in positives)
    comparisons.append((email, negative_score, wins, ties))

total_wins = sum(wins for _, _, wins, _ in comparisons)
total_ties = sum(ties for _, _, _, ties in comparisons)
pairs = len(positives) * len(negatives)
pair_auc = (total_wins + 0.5 * total_ties) / pairs
assert abs(pair_auc - roc_area) < 1e-12
rows = ''.join(
    f'<tr><th scope="row">{email}</th><td>{score:.4f}</td><td>{wins}</td>'
    f'<td>{ties}</td><td>{wins + 0.5 * ties:g}</td></tr>'
    for email, score, wins, ties in comparisons
)
panel(
    'Match the score ranking with the area',
    f'<p>{total_wins} wins, {pairs - total_wins - total_ties} losses, and {total_ties} tie.</p>'
    f'<p>AUC = ({total_wins} + 0.5 × {total_ties}) ÷ {pairs} = <strong>{pair_auc:.3f}</strong>.</p>'
    + disclosure(
        'Show the credit earned against each legitimate email',
        '<table><thead><tr><th scope="col">Legitimate email</th><th scope="col">Score</th>'
        '<th scope="col">Spam wins</th><th scope="col">Ties</th><th scope="col">Credit</th></tr></thead>'
        f'<tbody>{rows}</tbody></table>',
    ),
)

## 7. Read the full test fold

The same fitted model scores all **1,168 held-out emails**, including **445 recorded Spam**. These are the results on that fold using the same score rounding and decision rule.

| Cutoff | TP | FN | FP | TN |
| ---: | ---: | ---: | ---: | ---: |
| 0.20 | 435 | 10 | 169 | 554 |
| 0.50 | 400 | 45 | 81 | 642 |
| 0.75 | 347 | 98 | 31 | 692 |

ROC AUC on this fold is **0.962**. We chose the 20-classroom selection by label and score so we could count by hand. Use the complete fold for the benchmark on this 1999 dataset and evaluate a current mail filter on recent, representative data.

Revisit the [interactive Classification Metrics lesson](https://sharanharsoor.github.io/ml-learning-lab/classification-regression-metrics/#classification). Use the cutoff to pick a folder, the recorded label to count the kind of decision, and the metric that answers the question you are asking.

### Optional · Refit the model that made the scores

The main cells already contain the fitted model’s saved scores. For a reproducibility walkthrough, change `run_refit` to True. The cell loads UCI’s data directly. The lesson snapshot was prepared with scikit-learn **1.6.1**. Solver tolerances or library versions can change a score near a rounding boundary. The saved scores keep the browser lesson, worked examples, and notebook in sync.

[View the complete fixture generator](https://github.com/sharanharsoor/ml-learning-lab/blob/main/labs/classification-regression-metrics/scripts/prepare_spam_classification.py)

In [ ]:
run_refit = False
if run_refit:
    import os

    import numpy as np
    import pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    source = 'https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data'
    records = pd.read_csv(os.environ.get('UCI_SPAMBASE_FILE', source), header=None)
    X, y = records.iloc[:, :-1], records.iloc[:, -1]
    groups = X.groupby(X.columns.tolist(), sort=False, dropna=False).ngroup()
    split = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=37)
    train, test = next(split.split(X, y, groups))
    assert set(groups.iloc[train]).isdisjoint(groups.iloc[test])

    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1500))
    model.fit(X.iloc[train], y.iloc[train])
    spam_column = list(model.classes_).index(1)
    probabilities = np.round(model.predict_proba(X.iloc[test])[:, spam_column], 4)
    scored = {f'E{i + 1}': p for i, p in zip(test, probabilities)}
    refitted = [(email, scored[email], label) for email, _, label in emails]

    print(f'Fitted on {len(train):,} emails and scored {len(test):,} held-out emails.')
    print(f'ROC AUC on the complete fold: {roc_auc_score(y.iloc[test], probabilities):.3f}')
    print(f'Stored teaching scores matching this run: {sum(a == b for a, b in zip(emails, refitted))}/{len(emails)}')